# Lip Reading Project - Data Collection (Tasks API Edition)

Notebook ini didesain untuk mengumpulkan data video gerakan bibir menggunakan **MediaPipe Tasks API**. Kita akan mengekstrak landmark bibir secara real-time dan menyimpannya untuk training model GRU.

**Project Goal**: Isolated Word Lip Reading (20 Kata).
**Target Model**: Landmark-based RNN (GRU).

In [106]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import os
import json
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported. MediaPipe version:", mp.__version__)

✓ Libraries imported. MediaPipe version: 0.10.33


## Section 1: Setup Dataset & Word List

Berikut adalah daftar 20 kata yang akan kita kumpulkan datanya.

In [107]:
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "lip_reading_data"
MODEL_PATH = 'face_landmarker.task' # Pastikan file ini ada di root

WORDS = [
    'buka', 'tutup', 'antek-antek', 'tulis', 'menulis', 
    'berbicara', 'pintu', 'membawa', 'mengambil', 'meletakkan', 
    'mengirim', 'menyimpan', 'memperbaiki', 'pembelajaran', 'perbaikan', 
    'pemahaman', 'pemberitahuan', 'komunikasi', 'transformasi', 'buku', 'asing'
]

def create_dirs(word_list):
    for word in word_list:
        word_dir = DATASET_DIR / "words" / word
        for sub in ["videos", "lips", "landmarks", "metadata"]:
            (word_dir / sub).mkdir(parents=True, exist_ok=True)
    print(f"✓ Directory structure ready for {len(word_list)} words.")

create_dirs(WORDS)

✓ Directory structure ready for 21 words.


## Section 2: MediaPipe Tasks Initialization

Kita menggunakan `FaceLandmarker` dari API terbaru MediaPipe.

In [108]:
# Initialize MediaPipe Face Landmarker
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

# Lip indices for drawing/cropping
LIP_INDICES = [
    61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308
]

print("✓ MediaPipe Face Landmarker initialized.")

✓ MediaPipe Face Landmarker initialized.


W0000 00:00:1780551331.948009   79604 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1780551331.951904   79604 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780551331.953555   79623 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.1-arch1.2), renderer: Mesa Intel(R) Graphics (ADL GT2)
W0000 00:00:1780551331.955360   79605 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780551331.965934   79606 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## Section 3: Recording with Visual Feedback

Fungsi ini akan membuka webcam dan memberikan feedback titik bibir agar kamu tahu kalau bibirmu terdeteksi dengan benar.

In [109]:
def record_sample(word_label, duration=3, fps=30):
    cap = cv2.VideoCapture(0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    video_path = DATASET_DIR / "words" / word_label / "videos" / f"{word_label}_{timestamp}.mp4"
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(video_path), fourcc, fps, (width, height))
    
    print(f"Recording '{word_label.upper()}'... Get ready!")
    frame_count = 0
    total_frames = duration * fps
    
    while frame_count < total_frames:
        ret, frame = cap.read()
        if not ret: break
        
        # Save raw frame
        out.write(frame)
        
        # Processing for display (feedback)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        detection_result = detector.detect(mp_image)
        
        display_frame = frame.copy()
        if detection_result.face_landmarks:
            for idx in LIP_INDICES:
                pt = detection_result.face_landmarks[0][idx]
                cv2.circle(display_frame, (int(pt.x * width), int(pt.y * height)), 2, (0, 255, 0), -1)
        
        cv2.putText(display_frame, f"Word: {word_label} ({frame_count}/{total_frames})", (20, 50), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        cv2.imshow("Recording Data - Press 'q' to Cancel", display_frame)
        frame_count += 1
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Recording cancelled.")
            break
            
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"✓ Saved: {video_path}")
    return video_path

## Section 4: Start Collecting!

Jalankan cell di bawah untuk mulai merekam sampel.

In [110]:
# Tentukan kata dan jumlah sampel secara interaktif
WORD_TO_RECORD = input("Masukkan kata yang ingin direkam (contoh: buka, tutup, dll): ").lower().strip()

# Validasi apakah kata ada di daftar (opsional tapi disarankan agar folder rapi)
if WORD_TO_RECORD not in WORDS:
    print(f"⚠️ Peringatan: '{WORD_TO_RECORD}' tidak ada dalam daftar WORDS default.")
    create_new = input("Buat folder baru untuk kata ini? (y/n): ")
    if create_new.lower() == 'y':
        for sub in ["videos", "lips", "landmarks", "metadata"]:
            (DATASET_DIR / "words" / WORD_TO_RECORD / sub).mkdir(parents=True, exist_ok=True)
    else:
        print("Dibatalkan.")
        # Kita gunakan break/raise atau stop cell di sini jika perlu
        raise Exception("Proses dibatalkan oleh user.")

try:
    NUM_SAMPLES = int(input(f"Berapa banyak sampel untuk kata '{WORD_TO_RECORD}'? "))
except ValueError:
    print("Masukkan angka yang valid!")
    NUM_SAMPLES = 0

for i in range(NUM_SAMPLES):
    print(f"\nSample {i+1}/{NUM_SAMPLES}")
    input(f"Siap merekam '{WORD_TO_RECORD}'? Tekan Enter untuk mulai...")
    record_sample(WORD_TO_RECORD, duration=3)



Sample 1/5
Recording 'TUTUP'... Get ready!
✓ Saved: /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/LipReadingV.1/lip_reading_data/words/tutup/videos/tutup_20260604_123534.mp4

Sample 2/5
Recording 'TUTUP'... Get ready!
✓ Saved: /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/LipReadingV.1/lip_reading_data/words/tutup/videos/tutup_20260604_123539.mp4

Sample 3/5
Recording 'TUTUP'... Get ready!
✓ Saved: /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/LipReadingV.1/lip_reading_data/words/tutup/videos/tutup_20260604_123543.mp4

Sample 4/5
Recording 'TUTUP'... Get ready!
✓ Saved: /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/LipReadingV.1/lip_reading_data/words/tutup/videos/tutup_20260604_123547.mp4

Sample 5/5
Recording 'TUTUP'... Get ready!
✓ Saved: /home/takumifahri/Development/Jupyter_Dev/Visual_Komputer_Cerdas/Project Semester/LipReadingV.1/lip_rea

### Tip:
Setelah selesai merekam semua kata, jalankan script `extract_features.py` di terminal untuk memproses data sebelum di-upload ke Colab!